# 📊 Atelier ML — Analyse Comportementale Clientèle Retail
**Notebook d'Exploration & Prototypage**

Ce notebook sert à explorer le dataset et tester les transformations **avant** de les écrire dans les scripts de production (`src/`).

---
## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils import load_raw_data, basic_eda, plot_missing_values, plot_churn_distribution, get_highly_correlated_pairs

%matplotlib inline
sns.set_style('darkgrid')
pd.set_option('display.max_columns', 60)
print('Setup OK')

## 2. Chargement & EDA de base

In [ ]:
df = load_raw_data()
basic_eda(df)

In [ ]:
df.head(3)

## 3. Valeurs manquantes

In [ ]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
missing[missing > 0].plot(kind='bar', figsize=(8,4), color='steelblue', title='Missing values (%)')
plt.tight_layout()
plt.show()

## 4. Distribution de la variable cible (Churn)

In [ ]:
counts = df['Churn'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_xticklabels(['Loyal', 'Churned'], rotation=0)
axes[0].set_title('Churn counts')
axes[1].pie(counts, labels=['Loyal','Churned'], colors=['steelblue','tomato'],
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Churn ratio')
plt.tight_layout()
plt.show()
print(f'Imbalance ratio: {counts[1]/counts[0]:.2f}  →  SMOTE needed in training')

## 5. Distribution des features numériques clés

In [ ]:
num_features = ['Recency', 'Frequency', 'MonetaryTotal', 'Age', 'SatisfactionScore', 'CustomerTenureDays']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, num_features):
    df[col].dropna().hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Distribution des features numériques', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Features numériques vs Churn (boxplots)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, num_features):
    df.boxplot(column=col, by='Churn', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Churn (0=Loyal, 1=Churned)')
plt.suptitle('Boxplots: Features vs Churn', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Features catégorielles vs Churn

In [ ]:
cat_features = ['RFMSegment', 'CustomerType', 'SpendingCategory', 'Region', 'AccountStatus']
fig, axes = plt.subplots(1, len(cat_features), figsize=(18, 5))
for ax, col in zip(axes, cat_features):
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=ax, color='tomato', edgecolor='white')
    ax.set_title(f'Churn rate by\n{col}')
    ax.set_ylabel('Churn rate')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 8. Matrice de corrélation

In [ ]:
num_df = df.select_dtypes(include=[np.number]).drop(columns=['CustomerID','Churn'], errors='ignore')
corr = num_df.corr()
fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0, linewidths=.3, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
print('Highly correlated pairs (|corr| > 0.85):')
pairs = get_highly_correlated_pairs(df, threshold=0.85)
print(pairs.to_string(index=False))

## 9. Valeurs aberrantes

In [ ]:
print('SupportTicketsCount unique:', df['SupportTicketsCount'].value_counts().sort_index().head(20).to_dict())
print('\nSatisfactionScore unique: ', df['SatisfactionScore'].value_counts().sort_index().to_dict())
print('\nNewsletterSubscribed:', df['NewsletterSubscribed'].unique())

## 10. Parsing RegistrationDate

In [ ]:
sample_dates = df['RegistrationDate'].head(10)
print('Sample dates:')
print(sample_dates.tolist())

parsed = pd.to_datetime(sample_dates, dayfirst=True, errors='coerce')
print('\nParsed:')
print(parsed.tolist())

# Check overall parse success rate
all_parsed = pd.to_datetime(df['RegistrationDate'], dayfirst=True, errors='coerce')
nat_count = all_parsed.isna().sum()
print(f'\nNaT count: {nat_count} / {len(df)} ({nat_count/len(df)*100:.1f}%)')

## 11. RFM Analysis
Recency, Frequency, Monetary — the 3 pillars of customer value.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rfm_cols = ['Recency', 'Frequency', 'MonetaryTotal']
colors_by_churn = df['Churn'].map({0: 'steelblue', 1: 'tomato'})
for ax, col in zip(axes, rfm_cols):
    for churn_val, color, label in [(0,'steelblue','Loyal'), (1,'tomato','Churned')]:
        df[df['Churn']==churn_val][col].hist(bins=40, ax=ax, alpha=0.6, color=color, label=label)
    ax.set_title(col)
    ax.legend()
plt.suptitle('RFM Distribution by Churn Status')
plt.tight_layout()
plt.show()

---
## ✅ Summary & Next Steps

| Issue | Columns | Action |
|---|---|---|
| Constant feature | NewsletterSubscribed | Drop |
| Raw text date | RegistrationDate | Parse → RegYear/Month/Day/Weekday |
| Raw IP | LastLoginIP | Extract IP_IsPrivate, IP_FirstOctet |
| Sentinel values | SupportTicketsCount (-1, 999), SatisfactionScore (-1, 99) | → NaN |
| Missing Age (30%) | Age | KNN Imputation |
| Missing AvgDaysBetween | AvgDaysBetweenPurchases | Median imputation |
| High cardinality | Country (37+) | Target encoding |
| Class imbalance | Churn (67%/33%) | SMOTE in training |
| High correlation | Multiple pairs | Drop one of each pair (|r|>0.85) |

**Run the full pipeline:** `python src/preprocessing.py` → `python src/train_model.py`